In [ ]:
import torch
torch.cuda.is_available()

True

# HuggingFace Login

In [ ]:
!pip install huggingface_hub

In [1]:
from huggingface_hub import login

In [2]:
from google.colab import userdata
huggingface_api = userdata.get('huggingface_api')
login(huggingface_api)

In [ ]:
from huggingface_hub import whoami
whoami()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'type': 'user',
 'id': '68e0e65b71512dfe66eda6bf',
 'name': 'Maddala-Hemanth',
 'fullname': 'Maddala Hemanth',
 'email': 'hemanthmaddala04@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1782864000,
 'isPro': False,
 'avatarUrl': '/avatars/29927c0c59b1b3ec740c0e8280a80142.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'Finetuning',
   'role': 'read',
   'createdAt': '2026-06-07T06:06:36.870Z'}}}

# loading "model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0 model from HuggingFace_Hub'

- free to use "mistralai/Mistral-7B-Instruct-v0.3" if have GPU setup

In [3]:
!pip install -U bitsandbytes>=0.46.1 transformers

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [4]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [5]:
# Fix padding (important)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Quantization of model

In [6]:
# 4-bit quantization (VERY IMPORTANT)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16   # 🔥 ADD THIS
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

# Testing the Model

In [8]:
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU Memory Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU Memory Allocated: 0.73 GB
GPU Memory Reserved: 0.76 GB


In [9]:
## without finetuning
prompt = "Generate SQL: Get all users from users table"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Generate SQL: Get all users from users table
```
SELECT * FROM users;
```

2. Update a user:
```
UPDATE users SET name = 'John Doe' WHERE id = 1;
```

3. Delete a user:
```
DELETE FROM users WHERE id = 1;
```

4. Insert a new user:
```
INSERT INTO users (name, email, password) VALUES (?, ?, ?);
```

5. Update a user


In [10]:
prompt = """
You are an SQL generator.

Schema:
users(id, name, signup_date)

Question:
Get all users who signed up after 2023

SQL:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are an SQL generator.

Schema:
users(id, name, signup_date)

Question:
Get all users who signed up after 2023

SQL:
SELECT * FROM users WHERE signup_date >= DATEADD(year, -1, GETDATE());

Output:
| id | name | signup_date |
| --- | --- | --- |
| 1  | John | 2022-01-01 |
| 2  | Jane | 2023-01-01 |

Explanation:

1. The `DATEADD()` function


- models like Mistral Instruct are trained in a chat-style format.
- so converting dataset to
  - User: question
  - Assistant: answer

# Loading the dataset

In [11]:
!pip install datasets

In [12]:
from datasets import load_dataset

dataset = load_dataset("b-mc2/sql-create-context")

In [13]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 78577
    })
})


In [14]:
sample = dataset["train"][0]
print(sample)

{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}


In [15]:
schema_example = sample['context']
print(schema_example)

CREATE TABLE head (age INTEGER)


In [16]:
print(sample.keys())

dict_keys(['answer', 'question', 'context'])


# Dataset convertion to model style

In [17]:
def format_example(example):
    return {
        "messages": [
            {
                "role": "system",
                "content": f"""You are an SQL generator.
Return ONLY SQL query.

SCHEMA:
{example['context']}
"""
            },
            {
                "role": "user",
                "content": example["question"]
            },
            {
                "role": "assistant",
                "content": example["answer"]
            }
        ]
    }

In [18]:
formatted_dataset = dataset["train"].map(format_example)

In [19]:
print(formatted_dataset)

Dataset({
    features: ['answer', 'question', 'context', 'messages'],
    num_rows: 78577
})


In [20]:
sample = formatted_dataset[0]
print(sample)

{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)', 'messages': [{'content': 'You are an SQL generator.\nReturn ONLY SQL query.\n\nSCHEMA:\nCREATE TABLE head (age INTEGER)\n', 'role': 'system'}, {'content': 'How many heads of the departments are older than 56 ?', 'role': 'user'}, {'content': 'SELECT COUNT(*) FROM head WHERE age > 56', 'role': 'assistant'}]}


In [21]:
print(sample["messages"])

[{'content': 'You are an SQL generator.\nReturn ONLY SQL query.\n\nSCHEMA:\nCREATE TABLE head (age INTEGER)\n', 'role': 'system'}, {'content': 'How many heads of the departments are older than 56 ?', 'role': 'user'}, {'content': 'SELECT COUNT(*) FROM head WHERE age > 56', 'role': 'assistant'}]


# LORA config

In [14]:
!pip install peft trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 24.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [22]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [23]:
from peft import LoraConfig, get_peft_model

In [24]:
model.gradient_checkpointing_enable()
model.config.use_cache = False
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    #target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] = takes high memory, can cause memory usage limit -> use if GPU setup is there
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [25]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


# Preparing data for training

*   x_lables => full sequence -> [system + schema + question + SQL]
*   y_lables => only sql part (but previous part is also there , but making them to -100 -> helps for next token generation type models) [-100, -100, ..., SQL tokens]



In [26]:
sample = formatted_dataset[0]
print(sample)

{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)', 'messages': [{'content': 'You are an SQL generator.\nReturn ONLY SQL query.\n\nSCHEMA:\nCREATE TABLE head (age INTEGER)\n', 'role': 'system'}, {'content': 'How many heads of the departments are older than 56 ?', 'role': 'user'}, {'content': 'SELECT COUNT(*) FROM head WHERE age > 56', 'role': 'assistant'}]}


In [27]:
output_for_supervised_finetuning = sample["messages"][-1]["content"]
print(output_for_supervised_finetuning)

SELECT COUNT(*) FROM head WHERE age > 56


In [28]:
def tokenize(example):
    # Full chat text
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length"
    )

    labels = tokenized["input_ids"].copy()

    # Get assistant text
    assistant_text = example["messages"][-1]["content"]

    # Tokenize assistant separately
    assistant_tokens = tokenizer(
        assistant_text,
        add_special_tokens=False
    )["input_ids"]

    # Find assistant start index inside input_ids
    input_ids = tokenized["input_ids"]

    start_idx = -1
    for i in range(len(input_ids) - len(assistant_tokens)):
        if input_ids[i:i+len(assistant_tokens)] == assistant_tokens:
            start_idx = i
            break

    # Mask everything BEFORE assistant
    if start_idx != -1:
        for i in range(start_idx):
            labels[i] = -100

    tokenized["labels"] = labels

    return tokenized

In [29]:
# ── 1. Tokenize the dataset ──
tokenized_dataset = formatted_dataset.map(
    tokenize,
    remove_columns=formatted_dataset.column_names
)

Map:   0%|          | 0/78577 [00:00<?, ? examples/s]

In [30]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 78577
})

In [31]:
## neglect this if wanted to use whole dataset
small_dataset = tokenized_dataset.select(range(1000))

# Training the Model (Final FineTuning)

In [32]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./TinyLlama-sql-finetuned",
    num_train_epochs=1,   # start small
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    fp16=False,
    bf16=True,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,

    report_to="none",
    dataloader_pin_memory=False,
    optim="paged_adamw_8bit"
)

In [33]:
from trl import SFTTrainer
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=small_dataset,  # or full_dataset
    args=training_args,
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [34]:
print("Starting fine-tuning...")
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting fine-tuning...


Step,Training Loss
50,1.998774


Step,Training Loss
50,1.998774
100,0.678809


TrainOutput(global_step=125, training_loss=1.1884021301269532, metrics={'train_runtime': 1516.8542, 'train_samples_per_second': 0.659, 'train_steps_per_second': 0.082, 'total_flos': 3184942645248000.0, 'train_loss': 1.1884021301269532, 'entropy': 0.5826330342888832, 'num_tokens': 512000.0, 'mean_token_accuracy': 0.8618713024258614, 'epoch': 1.0})

# save the model

In [35]:
trainer.model.save_pretrained("tinyllama-sql-lora")
tokenizer.save_pretrained("tinyllama-sql-lora")

('tinyllama-sql-lora/tokenizer_config.json',
 'tinyllama-sql-lora/chat_template.jinja',
 'tinyllama-sql-lora/tokenizer.json')

# To goolge *drive*

In [36]:
from google.colab import drive
drive.mount('/content/drive')

trainer.model.save_pretrained("/content/drive/MyDrive/tinyllama-sql-lora")

Mounted at /content/drive


# Testing

In [39]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

model = PeftModel.from_pretrained(
    base_model,
    "tinyllama-sql-lora"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [38]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [53]:
prompt = """You are an SQL generator.
Return ONLY ONE SQL query.

SCHEMA:
CREATE TABLE users(id, name, age)

Now answer:

Question: Get all users names whose age is greater than or equal to 18

SQL:"""

In [54]:
model = model.to("cuda")

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

output = tokenizer.decode(outputs[0], skip_special_tokens=True)

sql = output.split("SQL:")[-1].strip().split("\n")[0]

print(sql)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SELECT name FROM users WHERE age >= 18
